# Assignment No. 4: Machine Learning Model Application & Evaluation
**Course:** Introduction to Data Science

**Project:** Student Performance — At-Risk Classification

**Student:** Habibullah Azam

## Problem Definition
This project uses student information to predict whether a student is at risk. A student with final grade `G3 < 10` is labelled `Yes`; otherwise, `No`. This is a classification task. `G3` is removed from the input features to prevent target leakage. This is a classroom exercise, not a tool for making real decisions about students.

## 1. Import libraries and load dataset
Run each code cell using the play button. The dataset is loaded from the UCI repository.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

url = "https://raw.githubusercontent.com/sachanganesh/student-performance-prediction/bd8e5514c5dd7d3b5d37ccf22fc74f35c67f0bf2/data/student-mat.csv"
df = pd.read_csv(url, sep=";")
print("Dataset shape:", df.shape)
display(df.head())

## 2. Explore the dataset
Check the dataset size, missing values, and final grade summary.

In [ ]:
print("Missing values:")
display(df.isnull().sum().to_frame("Missing Values"))
print("Final grade summary:")
display(df["G3"].describe())

## 3. Create target and prepare features
`At_Risk` is the target. We remove `G3` because it is used to create the target. Text columns are converted into numeric columns using one-hot encoding.

In [ ]:
df["At_Risk"] = np.where(df["G3"] < 10, "Yes", "No")
print("At-risk class counts:")
display(df["At_Risk"].value_counts().to_frame("Students"))
sns.countplot(data=df, x="At_Risk")
plt.title("At-Risk Student Distribution")
plt.show()

X = df.drop(columns=["G3", "At_Risk"])
y = df["At_Risk"]
X = pd.get_dummies(X, drop_first=True)
print("Feature shape:", X.shape)

## 4. Split the data and scale features
80% of the data is used for training and 20% for testing. The test data is kept separate until evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Train three machine learning models
We compare Logistic Regression, K-Nearest Neighbors (KNN), and Decision Tree.

In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_scaled, y_train)
pred_logistic = logistic_model.predict(X_test_scaled)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
pred_knn = knn_model.predict(X_test_scaled)

tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)
pred_tree = tree_model.predict(X_test)

predictions = {
    "Logistic Regression": pred_logistic,
    "KNN": pred_knn,
    "Decision Tree": pred_tree
}
print("All three models have been trained.")

## 6. Evaluate and compare models
Accuracy shows overall correctness. Precision tells how many predicted at-risk students were actually at risk. Recall tells how many actual at-risk students were found. F1-score balances precision and recall.

In [ ]:
results_list = []
for name, pred in predictions.items():
    results_list.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label="Yes", zero_division=0),
        "Recall": recall_score(y_test, pred, pos_label="Yes", zero_division=0),
        "F1 Score": f1_score(y_test, pred, pos_label="Yes", zero_division=0)
    })
results = pd.DataFrame(results_list)
display(results.round(3))
results.set_index("Model").plot(kind="bar", figsize=(10,5), ylim=(0,1))
plt.title("Comparison of Classification Models")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Confusion matrices
The confusion matrix compares actual and predicted classes for each model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=["Yes", "No"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["At Risk", "Not At Risk"],
                yticklabels=["At Risk", "Not At Risk"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 8. Check overfitting and underfitting
A large difference between training and test accuracy may indicate overfitting. Low accuracy on both may indicate underfitting. This is a simple check.

In [ ]:
fit_check = pd.DataFrame([
    {"Model": "Logistic Regression", "Training Accuracy": logistic_model.score(X_train_scaled, y_train), "Testing Accuracy": accuracy_score(y_test, pred_logistic)},
    {"Model": "KNN", "Training Accuracy": knn_model.score(X_train_scaled, y_train), "Testing Accuracy": accuracy_score(y_test, pred_knn)},
    {"Model": "Decision Tree", "Training Accuracy": tree_model.score(X_train, y_train), "Testing Accuracy": accuracy_score(y_test, pred_tree)}
])
fit_check["Train-Test Gap"] = fit_check["Training Accuracy"] - fit_check["Testing Accuracy"]
display(fit_check.round(3))

## 9. Conclusion
The following cell chooses the model with the highest recall for the at-risk class. Review precision and F1-score too before deciding which model is best for this task.

In [ ]:
best_row = results.loc[results["Recall"].idxmax()]
print("Model with highest recall:", best_row["Model"])
print(results.sort_values("Recall", ascending=False).round(3).to_string(index=False))
print("\nReason: Recall measures the proportion of actual at-risk students correctly identified.")
print("Limitations: This is a small dataset from specific schools, and results may change with another train-test split.")
print("The model shows patterns; it does not prove causes and should not be used alone for important decisions.")

## Submission checklist
1. Open the notebook in Google Colab.
2. Select **Runtime → Run all** and wait until every cell finishes.
3. Confirm the metric table and graphs appear.
4. Save the notebook with its outputs if your instructor requires them.
5. Submit this GitHub file link.

Be ready to explain: features (`X`), target (`y`), train-test split, accuracy, precision, recall, F1-score, confusion matrix, and overfitting.